# TE (Tight End) Round Regression (Ridge)

Predict draft round 1–8 (8 = undrafted) for tight ends using combine + RAS + PFF receiving + PFF run blocking grades.

- **Train**: 2015–2023 from `te_training.csv`
- **Test**: `te_testing.csv` filtered to 2024/2025 (drafted only); 2026 predictions for all prospects.


In [1]:
import numpy as np
import pandas as pd
import os
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

pd.set_option('display.max_rows', 200)
pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

# TE uses same receiving features as WR + run blocking grade
FEATURES_WITH_COLLEGE_TE = [
    # Physical / Athletic
    'Height', 'Weight', '40yd', 'Vertical', 'Broad Jump',
    'speed_score', 'explosive_score', 'RAS', 'arm_length_inches',
    # Receiving Efficiency
    'yprr', 'yards_per_reception', 'caught_percent',
    'avg_depth_of_target', 'targeted_qb_rating',
    # Ball Skills
    'contested_catch_rate', 'drop_rate',
    # YAC / Open Field
    'yards_after_catch_per_reception', 'avoided_tackles',
    # Alignment / Usage
    'slot_rate', 'wide_rate', 'inline_rate', 'route_rate',
    # Blocking
    'grades_run_block',
    # Experience / Context
    'player_game_count', 'p4_conference',
]

CONTAINS_WITH_COLLEGE_TE = [
    'contains_height', 'contains_weight', 'contains_40yd',
    'contains_vertical', 'contains_broad_jump',
    'contains_speed_score', 'contains_explosive_score',
    'contains_ras', 'contains_arm_length_inches',
    'contains_yprr', 'contains_yards_per_reception', 'contains_caught_percent',
    'contains_avg_depth_of_target', 'contains_targeted_qb_rating',
    'contains_contested_catch_rate', 'contains_drop_rate',
    'contains_yards_after_catch_per_reception', 'contains_avoided_tackles',
    'contains_slot_rate', 'contains_wide_rate', 'contains_inline_rate', 'contains_route_rate',
    'contains_grades_run_block',
    'contains_player_game_count', 'contains_p4_conference',
]

FEATURES_ALL = FEATURES_WITH_COLLEGE_TE + CONTAINS_WITH_COLLEGE_TE

In [2]:
P4_PRE_2024 = {
    'Alabama', 'Arkansas', 'Auburn', 'Florida', 'Georgia', 'Kentucky',
    'LSU', 'Mississippi', 'Mississippi State', 'Missouri', 'South Carolina',
    'Tennessee', 'Texas A&M', 'Vanderbilt',
    'Illinois', 'Indiana', 'Iowa', 'Maryland', 'Michigan', 'Michigan State',
    'Minnesota', 'Nebraska', 'Northwestern', 'Ohio State', 'Penn State',
    'Purdue', 'Rutgers', 'Wisconsin',
    'Baylor', 'Iowa State', 'Kansas', 'Kansas State', 'Oklahoma',
    'Oklahoma State', 'TCU', 'Texas', 'Texas Tech', 'West Virginia',
    'Cincinnati', 'Houston', 'UCF', 'BYU',
    'Boston College', 'Clemson', 'Duke', 'Florida State', 'Georgia Tech',
    'Louisville', 'Miami', 'North Carolina', 'North Carolina State',
    'Pittsburgh', 'Syracuse', 'Virginia', 'Virginia Tech', 'Wake Forest',
    'Arizona', 'Arizona State', 'California', 'Colorado', 'Oregon',
    'Oregon State', 'Stanford', 'UCLA', 'USC', 'Utah', 'Washington', 'Washington State',
}
P4_2024_PLUS = (P4_PRE_2024
    - {'Arizona', 'Arizona State', 'California', 'Colorado', 'Oregon', 'Oregon State',
       'Stanford', 'UCLA', 'USC', 'Utah', 'Washington', 'Washington State'}
    | {'Oregon', 'Washington', 'UCLA', 'USC', 'Arizona', 'Arizona State', 'Utah', 'Colorado', 'SMU'}
)
P4_2024_PLUS -= {'Oregon State', 'Washington State', 'California', 'Stanford'}

def get_p4(school, year):
    ref = P4_PRE_2024 if year < 2024 else P4_2024_PLUS
    return 1 if str(school).strip() in ref else 0


def add_engineered_features(df):
    df = df.copy()
    if df['Height'].dtype == object or df['Height'].astype(str).str.contains('-', na=False).any():
        def _ht(h):
            if pd.isna(h): return np.nan
            s = str(h).strip()
            if '-' in s:
                p = s.split('-')
                try: return int(p[0]) * 12 + int(p[1])
                except: return np.nan
            try: return float(s)
            except: return np.nan
        df['Height'] = df['Height'].apply(_ht)
    else:
        df['Height'] = pd.to_numeric(df['Height'], errors='coerce')

    for c in ['Weight', '40yd', 'Vertical', 'Broad Jump', 'RAS', 'arm_length_inches',
              'yprr', 'yards_per_reception', 'caught_percent', 'avg_depth_of_target',
              'targeted_qb_rating', 'contested_catch_rate', 'drop_rate',
              'yards_after_catch_per_reception', 'avoided_tackles',
              'slot_rate', 'wide_rate', 'inline_rate', 'route_rate',
              'grades_run_block', 'player_game_count']:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')

    w = df['Weight']; s = df['40yd']
    df['speed_score'] = np.where((s > 0) & (~w.isna()), w * 200 / (s ** 4), np.nan)

    vert = df['Vertical']
    v_mean = vert.mean(); v_std = vert.std()
    df['explosive_score'] = np.where(~vert.isna(), (vert - v_mean) / (v_std + 1e-8), np.nan)

    df['p4_conference'] = df.apply(lambda r: get_p4(r['School'], int(r['Year'])), axis=1)

    flag_map = {
        'contains_height': 'Height', 'contains_weight': 'Weight',
        'contains_40yd': '40yd', 'contains_vertical': 'Vertical',
        'contains_broad_jump': 'Broad Jump',
        'contains_speed_score': 'speed_score', 'contains_explosive_score': 'explosive_score',
        'contains_ras': 'RAS', 'contains_arm_length_inches': 'arm_length_inches',
        'contains_yprr': 'yprr', 'contains_yards_per_reception': 'yards_per_reception',
        'contains_caught_percent': 'caught_percent',
        'contains_avg_depth_of_target': 'avg_depth_of_target',
        'contains_targeted_qb_rating': 'targeted_qb_rating',
        'contains_contested_catch_rate': 'contested_catch_rate',
        'contains_drop_rate': 'drop_rate',
        'contains_yards_after_catch_per_reception': 'yards_after_catch_per_reception',
        'contains_avoided_tackles': 'avoided_tackles',
        'contains_slot_rate': 'slot_rate', 'contains_wide_rate': 'wide_rate',
        'contains_inline_rate': 'inline_rate', 'contains_route_rate': 'route_rate',
        'contains_grades_run_block': 'grades_run_block',
        'contains_player_game_count': 'player_game_count',
    }
    for flag, col in flag_map.items():
        df[flag] = df[col].notna().astype(int) if col in df.columns else 0
    df['contains_p4_conference'] = 1
    return df

In [3]:
# ── Load and prepare training data ──────────────────────────────────────────
df = pd.read_csv('../data/processed/te_training.csv')
df = df[df['Year'].between(2015, 2023)].copy()
print(f'Train (2015–2023 TEs): {len(df)}')

df = add_engineered_features(df)
for c in FEATURES_ALL:
    if c not in df.columns:
        df[c] = 0

y = np.where(df['Drafted'].astype(bool), np.clip(df['Round'].fillna(1).astype(int), 1, 7), 8)
X_raw = df[FEATURES_ALL].copy()

imputer = KNNImputer(n_neighbors=10)
X = imputer.fit_transform(X_raw)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
ridge = Ridge(alpha=1.0, random_state=42)
ridge.fit(X_scaled, y)

y_pred_train = np.clip(ridge.predict(X_scaled), 1, 8)
print(f'Train MAE: {mean_absolute_error(y, y_pred_train):.4f}')
print(f'Train samples: {len(y)}')

Train (2015–2023 TEs): 181
Train MAE: 1.4495
Train samples: 181


In [4]:
# ── Load testing data ─────────────────────────────────────────────────────
te_testing = pd.read_csv('../data/processed/te_testing.csv')

te_2024 = te_testing[(te_testing['Year'] == 2024) & (pd.to_numeric(te_testing['Round'], errors='coerce') < 8)].copy()
te_2025 = te_testing[(te_testing['Year'] == 2025) & (pd.to_numeric(te_testing['Round'], errors='coerce') < 8)].copy()
te_2026 = te_testing[te_testing['Year'] == 2026].copy()

print(f'TE 2024 drafted: {len(te_2024)}, 2025: {len(te_2025)}, 2026 prospects: {len(te_2026)}')

def prepare_te_df(ldf, year):
    ldf = ldf.copy(); ldf['Year'] = year
    ldf = add_engineered_features(ldf)
    for c in FEATURES_ALL:
        if c not in ldf.columns: ldf[c] = 0
    return ldf

def eval_metrics(actual, pred, label):
    mae = mean_absolute_error(actual, pred)
    rmse = np.sqrt(mean_squared_error(actual, pred))
    r2 = r2_score(actual, pred)
    exact = (np.round(pred) == actual).mean()
    w1 = (np.abs(np.round(pred) - actual) <= 1).mean()
    print(f'{label} (n={len(actual)}): MAE={mae:.4f}, RMSE={rmse:.4f}, R²={r2:.4f}, '
          f'Exact={exact:.2%}, Within-1={w1:.2%}')

TE 2024 drafted: 12, 2025: 14, 2026 prospects: 27


In [5]:
te_2024 = prepare_te_df(te_2024, 2024)
te_2025 = prepare_te_df(te_2025, 2025)

pred_24 = np.clip(ridge.predict(scaler.transform(imputer.transform(te_2024[FEATURES_ALL]))), 1, 8)
pred_25 = np.clip(ridge.predict(scaler.transform(imputer.transform(te_2025[FEATURES_ALL]))), 1, 8)

actual_24 = pd.to_numeric(te_2024['Round'], errors='coerce').fillna(8).astype(int).values
actual_25 = pd.to_numeric(te_2025['Round'], errors='coerce').fillna(8).astype(int).values

eval_metrics(actual_24, pred_24, '2024 TEs')
eval_metrics(actual_25, pred_25, '2025 TEs')

2024 TEs (n=12): MAE=1.4901, RMSE=1.9286, R²=-0.0976, Exact=33.33%, Within-1=58.33%
2025 TEs (n=14): MAE=1.5286, RMSE=1.7922, R²=0.2611, Exact=14.29%, Within-1=42.86%


In [6]:
def te_tier(p):
    if p < 1.75: return 'Round 1'
    if p < 2.75: return 'Round 2'
    if p < 3.75: return 'Round 3'
    if p < 4.75: return 'Round 4'
    if p < 5.75: return 'Round 5'
    if p < 6.75: return 'Round 6'
    if p < 7.75: return 'Round 7'
    return 'UDFA'

d24 = te_2024[['Round', 'Pick', 'Player', 'School']].copy()
d24['predicted_round'] = pred_24; d24['tier'] = [te_tier(x) for x in pred_24]
d24['Round'] = pd.to_numeric(d24['Round'], errors='coerce').astype('Int64')
print('2024 TEs')
display(d24.sort_values('predicted_round').reset_index(drop=True))

d25 = te_2025[['Round', 'Pick', 'Player', 'School']].copy()
d25['predicted_round'] = pred_25; d25['tier'] = [te_tier(x) for x in pred_25]
d25['Round'] = pd.to_numeric(d25['Round'], errors='coerce').astype('Int64')
print('2025 TEs')
display(d25.sort_values('predicted_round').reset_index(drop=True))

2024 TEs


,Round,Pick,Player,School,predicted_round,tier
0,2,53.00,Ben Sinnott,KANSAS ST,2.88,Round 3
1,4,107.00,Theo Johnson,PENN STATE,3.30,Round 3
2,3,82.00,Tip Reiman,ILLINOIS,3.39,Round 3
3,7,246.00,Devin Culp,WASHINGTON,3.43,Round 3
4,7,231.00,Jaheim Bell,FLORIDA ST,3.67,Round 3
5,1,13.00,Brock Bowers,GEORGIA,3.99,Round 4
6,4,123.00,Cade Stover,Ohio State,4.11,Round 4
7,4,121.00,A.J. Barner,MICHIGAN,4.40,Round 4
8,4,101.00,Ja'Tavion Sanders,TEXAS,4.91,Round 5
9,7,194.00,Tanner McLachlan,ARIZONA,5.14,Round 5


2025 TEs


,Round,Pick,Player,School,predicted_round,tier
0,3,67.00,Harold Fannin Jr,Bowling Green,1.97,Round 2
1,1,14.00,Tyler Warren,Penn State,3.08,Round 3
2,2,46.00,Terrance Ferguson,Oregon,3.56,Round 3
3,2,42.00,Mason Taylor,LSU,3.62,Round 3
4,5,165.00,Oronde Gadsden II,Syracuse,3.91,Round 4
5,7,219.00,Thomas Fidone II,Nebraska,3.95,Round 4
6,1,10.00,Colston Loveland,Michigan,4.39,Round 4
7,5,163.00,Mitchell Evans,Notre Dame,4.88,Round 5
8,7,255.00,Luke Lachey,Iowa,5.00,Round 5
9,7,248.00,Moliki Matavao,UCLA,5.08,Round 5


In [7]:
# ── 2026 TE predictions ───────────────────────────────────────────────────
te_2026 = prepare_te_df(te_2026, 2026)
pred_26 = np.clip(ridge.predict(scaler.transform(imputer.transform(te_2026[FEATURES_ALL]))), 1, 8)

d26 = te_2026[['Player', 'School']].copy()
d26['Pos'] = 'TE'
d26['predicted_round'] = pred_26
d26['tier'] = [te_tier(x) for x in pred_26]
d26_sorted = d26.sort_values('predicted_round').reset_index(drop=True)

print(f'2026 TE predictions (n={len(pred_26)})')
display(d26_sorted)

d26[['Player', 'School', 'Pos', 'predicted_round']].to_csv(
    '../data/processed/te_2026_predictions.csv', index=False
)
print('Saved te_2026_predictions.csv')

2026 TE predictions (n=27)


,Player,School,Pos,predicted_round,tier
0,Kenyon Sadiq,Oregon,TE,3.41,Round 3
1,Dallen Bentley,Utah,TE,3.70,Round 3
2,Eli Stowers,Vanderbilt,TE,3.74,Round 3
3,Riley Nowakowski,Indiana,TE,4.04,Round 4
4,Michael Trigg,Baylor,TE,4.17,Round 4
5,Dae'Quan Wright,Mississippi,TE,4.53,Round 4
6,Matthew Hibner,SMU,TE,4.62,Round 4
7,Eli Raridon,Notre Dame,TE,4.64,Round 4
8,Justin Joly,NC State,TE,4.78,Round 5
9,DJ Rogers,TCU,TE,4.78,Round 5


Saved te_2026_predictions.csv


In [8]:
# ── Feature coefficients ─────────────────────────────────────────────────
coef_df = pd.DataFrame({'feature': FEATURES_ALL, 'coefficient': ridge.coef_})
coef_df = coef_df.reindex(coef_df['coefficient'].abs().sort_values(ascending=False).index)
print('Top 20 features by |coefficient|:')
display(coef_df.head(20).reset_index(drop=True))

Top 20 features by |coefficient|:


,feature,coefficient
0,speed_score,-2.32
1,40yd,-2.10
2,contains_yards_after_catch_per_reception,1.20
3,contains_yards_per_reception,1.20
4,RAS,-0.75
5,yprr,-0.59
6,player_game_count,-0.49
7,Weight,0.38
8,drop_rate,0.38
9,targeted_qb_rating,0.34
